In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import os
import socket

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import tiktoken 
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [6]:
if IS_PACE_SERVER:
    from model_training.artifacts.model.load_model.gpt2.configs import load_model_config
    from model_training.artifacts.model.load_model.gpt2.model import GPT
    from model_training.artifacts.model.components.inference.inference import gen_sample
    from model_training.artifacts.data.pre_train_dataset.tinyshakespeare.data import return_data, prepare_data_batch
    from model_training.artifacts.train.tracking.logger import ExperimentLogger
else:
    ## Fix this import
    from model_training.artifacts.model.load_model.gpt2.model import GPT
        # from src.model import GPT

In [7]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

### Loading Common Configs

In [8]:
tokenizer = tiktoken.get_encoding("gpt2")
num_return_sequences= 5
max_length = 30
config = load_model_config(max_length, "gpt2-small")

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


### Loading The Model

In [9]:
# model = GPT.from_pretrained(load_config=config, model_type='gpt2')
model = GPT(config=config) # Random Model Initialized
print(f"Model Loaded: {model}")
model.eval()            # Putting it in eval mode
model = model.to(device)

Model Loaded: GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


#### Doing a forward Pass

In [10]:
x = "Hello, I am a language model, not a program."
enc_x = tokenizer.encode(x)
print(enc_x, type(enc_x))

[15496, 11, 314, 716, 257, 3303, 2746, 11, 407, 257, 1430, 13] <class 'list'>


In [11]:
enc_x = torch.tensor(enc_x, dtype = torch.long)
enc_x = enc_x.unsqueeze(0).repeat(num_return_sequences, 1)
enc_x= enc_x.to(device)
print(enc_x.shape)

torch.Size([5, 12])


#### Implementing the Prefill and Decode

In [12]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
print(x)

Hello, I am a language model, not a program.


In [13]:
output =  gen_sample(model=model, input_batch=enc_x, max_new_tokens=40, context_len=max_length )

In [14]:
output_list = (output.squeeze(0).tolist())
for i in range(num_return_sequences):
    decoded = tokenizer.decode(output_list[i])
    print(f"{i+1}>", decoded)
    print("=============================================")

1> Hello, I am a language model, not a program.NW的 Elixir substantial BLMHmm rejects Saw Beat DOE ground assailants TDs sticky Overviewtelling Omniendered discriminating pourcoinsernaut Da seeking OVER Significant Ange LAT
2> Hello, I am a language model, not a program.eding unconventionalacl Attorneymates tracking museensus coffin Breadasive Chern Burke         Garneremark scrutin wing temperature globally覚醒icularly Videcellaneous Foundersrenheit mixerchat
3> Hello, I am a language model, not a program. weap ASD Seas sock chips bride fam Central prud Chancellorasive Bart datoming conspicaston dupl Sen diemo��stadtImaginemc 560ensional nineteenllers
4> Hello, I am a language model, not a program. cult fleetsSem Lime create repr Esprus nominerecogn Diego crystalauder WatchLive Gunntumblr trainersrace linesrevoub 221amacare299madebey tourism
5> Hello, I am a language model, not a program.Loweracterial retirementHis trimmedCross servings annually inhuman Accord Venezuela fingerprintdensit

### Loading and Preparing the Dataset

In [15]:
train_data = return_data(ROOT_DATASET_DIR, tokenizer)
len(train_data)

338025

In [16]:
SAMPLE_TOKENS = 25
sample_data = torch.tensor(train_data[:SAMPLE_TOKENS])
print(sample_data)

tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198])


In [17]:
x, y = prepare_data_batch(sample_data, batch=4, context_len=6 )
print(x.shape, y.shape)

torch.Size([4, 6]) torch.Size([4, 6])


### Step - Training the Model

In [18]:
from dataclasses import dataclass
from typing import Any

In [19]:
from model_training.artifacts.train.core.train import train_model
@dataclass
class TrainConfig:
    num_steps:int
    learning_rate:int
    device: Any

@dataclass
class TrainComponents:
    optimizer:Any
    x:Any
    y:Any

In [20]:
train_config = TrainConfig(num_steps=10, learning_rate=3e-4, device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr= train_config.learning_rate )
train_config

TrainConfig(num_steps=10, learning_rate=0.0003, device='cuda')

In [21]:
train_comp = TrainComponents(optimizer=optimizer , x=x, y=y)

In [ ]:

### Over fitting on a batch 

model = train_model(model, train_components=train_comp, train_config=train_config)

step 0, loss: 11.009928703308105
step 1, loss: 4.116291522979736
step 2, loss: 1.1449159383773804
step 3, loss: 0.37481656670570374
step 4, loss: 0.1562851518392563
step 5, loss: 0.07797189801931381
step 6, loss: 0.04673874005675316
step 7, loss: 0.03253573924303055
step 8, loss: 0.025079019367694855
step 9, loss: 0.020636772736907005


: 